In [2]:
# zip_path="/mnt/2TB_WD/rishi/aurn.zip"
# unzip_dir="/mnt/2TB_WD/rishi/aurn"
# import zipfile, os

# os.makedirs(unzip_dir, exist_ok=True)
# with zipfile.ZipFile(zip_path, 'r') as zf:
#     zf.extractall(unzip_dir)

In [3]:
import rdata
import pandas as pd
import os
from tqdm import tqdm

def rdata_to_dataframe(file_path):
    parsed = rdata.parser.parse_file(fr"{file_path}")
    converted = rdata.conversion.convert(parsed)
    col_name=list(converted.keys())[0]
    df=pd.DataFrame(converted[col_name])
    df["date"] = pd.to_datetime(df["date"], unit="s", utc=True)
    return df

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

# input_dir = "/mnt/2TB_WD/rishi/aurn/aurn_data"
# output_dir = "/mnt/2TB_WD/rishi/aurn_csv"
# os.makedirs(output_dir, exist_ok=True)
# workers=24
# def process_file(fname):
#     fpath = os.path.join(input_dir, fname)
#     df = rdata_to_dataframe(fpath)
#     out_path = os.path.join(output_dir, fname.rsplit(".", 1)[0] + ".csv")
#     df.to_csv(out_path, index=False)
#     return out_path

# files = [f for f in os.listdir(input_dir) if f.endswith((".RData", ".rdata", ".rds"))]

# with ProcessPoolExecutor(max_workers=workers) as executor:
#     futures = {executor.submit(process_file, f): f for f in files}
#     for future in tqdm(as_completed(futures), total=len(futures)):
#         try:
#             future.result()
#         except Exception as e:
#             print(f"error with file {futures[future]}: {e}")

In [ ]:
from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

csv_dir    = "/mnt/2TB_WD/rishi/aurn/aurn_csv"
proc_dir   = "/mnt/2TB_WD/rishi/aurn/aurn_processed"
date_start = "2022-06-30"
date_end   = "2025-12-31"
os.makedirs(proc_dir, exist_ok=True)

# AURN RData column names (lowercase) -> (output col label, filename suffix) per CNEMC convention
AURN_POL_MAP = {
    "co":    ("CO (mg/m³)",    "CO"),
    "no2":   ("NO2 (µg/m³)",   "NO2"),
    "o3":    ("Ozone (µg/m³)", "Ozone"),
    "pm10":  ("PM10 (µg/m³)",  "PM10"),
    "pm2.5": ("PM2.5 (µg/m³)", "PM2.5"),
    "so2":   ("SO2 (µg/m³)",   "SO2"),
}

# Only include yearly CSVs whose year falls within the date range
year_start = pd.to_datetime(date_start).year
year_end   = pd.to_datetime(date_end).year

site_files = {}
for f in os.listdir(csv_dir):
    if not f.endswith(".csv"):
        continue
    parts = f[:-4].rsplit("_", 1)
    if len(parts) != 2 or not parts[1].isdigit():
        print("error with ", f)
    site, year = parts[0], int(parts[1])
    if year_start <= year <= year_end:
        site_files.setdefault(site, []).append(f)

full_index = pd.date_range(
    start=date_start, end=date_end, freq="h", name="Timestamp", tz="UTC"
)


def process_site_aurn(site, files):
    dfs = [pd.read_csv(os.path.join(csv_dir, f)) for f in sorted(files)]
    combined = pd.concat(dfs, ignore_index=True)
    combined["date"] = pd.to_datetime(combined["date"], utc=True, errors="coerce")
    combined = (combined.dropna(subset=["date"])
                        .set_index("date")
                        .rename_axis("Timestamp")
                        .sort_index()
                        .loc[date_start:date_end])
    if combined.empty:
        return site, 0
    combined.columns = combined.columns.str.lower().str.strip()

    saved = 0
    for aurn_col, (out_col, pol_name) in AURN_POL_MAP.items():
        if aurn_col not in combined.columns or combined[aurn_col].isna().all():
            continue
        out = (combined[[aurn_col]]
               .rename(columns={aurn_col: out_col})
               .reindex(full_index))
        out.to_csv(os.path.join(proc_dir, f"site_{site}_{pol_name}.csv"))
        saved += 1
    return site, saved


sites_processed = 0
with ProcessPoolExecutor() as executor:
    futures = {executor.submit(process_site_aurn, site, files): site
               for site, files in site_files.items()}
    for future in tqdm(as_completed(futures), total=len(futures)):
        site, n = future.result()
        if n:
            sites_processed += 1

print(f"Done. {sites_processed}/{len(site_files)} sites had data in range.")

100%|██████████| 210/210 [00:08<00:00, 23.37it/s]

Done. 209/210 sites had data in range.
